# 00 — Préparation des données

**Objectif :** créer des versions propres du dataset :
- `data/dataset_v1.csv` : uniquement les colonnes **claires** (on sait ce qu'elles mesurent) et **utiles** (identification, cible, variables explicatives) ;
- `data/dataset_v2.csv` : la v1 avec **une ligne par site** (mesures d'un même point GPS agrégées).

**Règle :** le fichier original `data/dataset.csv` n'est jamais modifié.

Chaque étape suit la structure : **Objectif** → **Raisonnement** → **Décision** (ou **Constat** après le code).

In [1]:
# Import de pandas pour manipuler le tableau de données
import pandas as pd

In [2]:
# Chargement du dataset original (173 mesures de Q10)
df = pd.read_csv("../data/dataset.csv")
df.head()

,site,source,study,Q10,lat,lon,MAT_public,MAP_public,pH_public,N515,Ts_max,Ts_mean,SWC_warm_rel,new,cluster
0,paper_2,paper,paper_2,3.209642,42.412050,117.247400,1.422987,573.889280,7.160501,2401.352731,17.621953,3.469722,0.998680,False,c0
1,paper_3,paper,paper_3,2.779197,48.666667,7.083333,10.181837,1080.277650,5.780671,2812.636121,19.346878,10.429616,0.847827,False,c1
2,paper_4,paper,paper_4,1.567165,49.542000,-89.692000,2.358450,774.421208,5.445515,4037.599945,18.075066,6.053283,0.934605,False,c2
3,paper_5,paper,paper_4,2.046868,49.542000,-89.692000,2.358450,774.421208,5.445515,4037.599945,18.075066,6.053283,0.934605,False,c2
4,paper_6,paper,paper_4,2.175395,49.542000,-89.692000,2.358450,774.421208,5.445515,4037.599945,18.075066,6.053283,0.934605,False,c2


## Étape 1 — Inventaire des colonnes

**Objectif :** décider, colonne par colonne, ce que l'on garde.

**Raisonnement :** une variable explicative n'est utile que si (1) on sait ce qu'elle mesure et (2) elle existe sous forme de carte mondiale. Sans cela, on ne pourra pas produire la carte de Q10 destinée à ORCHIDEE.

| Colonne | Signification | Statut | Décision |
|---|---|---|---|
| `site` | identifiant unique de la mesure | clair | garder |
| `source` | origine de la mesure (`paper` = Hacopian, `mgrsd`, `holisoils`, …) | clair | garder |
| `study` | étude / publication d'origine | clair | garder (utile pour grouper la validation croisée) |
| `Q10` | variable cible | clair | garder |
| `lat`, `lon` | coordonnées du site | clair | garder |
| `MAT_public` | température moyenne annuelle de l'air à 2 m (°C), ERA5-Land* | clair | garder |
| `MAP_public` | précipitations moyennes annuelles (mm/an), ERA5-Land* | clair | garder |
| `pH_public` | pH du sol (eau), SoilGrids 0–5 cm* | clair | garder |
| `N515` | azote du sol SoilGrids, 5–15 cm (mg/kg, unité vérifiée dans `03_enrichissement_donnees.ipynb`) | clair (confirmé) | garder |
| `Ts_max`, `Ts_mean` | température du sol max / moyenne : source et période inconnues | pas clair | retirer |
| `SWC_warm_rel` | probablement l'humidité du sol en saison chaude rapportée à la moyenne annuelle** : définition exacte inconnue | pas clair | retirer |
| `new` | sens inconnu : `True` pour `root_removal` et `figure`, mais `False` pour `new paper` | pas clair | retirer |
| `cluster` | méthode de construction inconnue | pas clair | retirer |

\* Mêmes sources que Hacopian (vérifié dans son script R publié : `github.com/melaniehacopian/microbial_Q10_workflow`). Pour MAT et MAP, c'est confirmé par recalcul dans `03_enrichissement_donnees.ipynb` (r > 0.99).

\*\* Introuvable en ligne ; hypothèse déduite des valeurs, que l'on ne pouvait pas vérifier à ce stade. **Mise à jour :** la définition est confirmée dans `03_enrichissement_donnees.ipynb` (humidité ERA5-Land 0–7 cm, 3 mois les plus chauds / moyenne annuelle ; r = 0.98). La variable y est recalculée et réintégrée en v3.

**Décision :** on garde 10 colonnes. Les colonnes retirées pourront revenir dans une version ultérieure si leur origine est retrouvée. Pour `cluster`, on construira nos propres groupes, de façon transparente, au moment de la validation croisée.

In [3]:
# Sélection des colonnes claires et utiles (cf. tableau ci-dessus)
colonnes = ["site", "source", "study", "Q10", "lat", "lon", "MAT_public", "MAP_public", "pH_public", "N515"]
df_v1 = df[colonnes]

## Étape 2 — Vérifications

**Objectif :** s'assurer que la version 1 est complète et cohérente avant de la sauvegarder.

**Raisonnement :** on vérifie l'absence de valeurs manquantes, puis que les plages de valeurs sont physiquement plausibles (ex. pH entre 0 et 14).

In [4]:
# Nombre de valeurs manquantes par colonne
df_v1.isna().sum()

site          0
source        0
study         0
Q10           0
lat           0
lon           0
MAT_public    0
MAP_public    0
pH_public     0
N515          0
dtype: int64

In [5]:
# Statistiques descriptives pour vérifier la plausibilité des plages de valeurs
df_v1.describe().round(2)

,Q10,lat,lon,MAT_public,MAP_public,pH_public,N515
count,173.00,173.00,173.00,173.00,173.00,173.00,173.00
mean,2.44,37.87,64.01,11.22,1212.35,6.00,2763.30
std,0.96,11.88,75.89,6.37,528.52,0.81,1379.18
min,0.42,-37.42,-125.32,-5.19,229.65,4.44,837.32
25%,1.90,29.93,11.64,7.08,774.42,5.43,1833.92
50%,2.27,36.42,106.28,11.02,1092.40,5.95,2276.14
75%,2.89,47.18,117.43,16.32,1635.94,6.32,3427.60
max,6.04,64.23,144.09,24.52,3386.24,8.25,9327.54


**Constat :** aucune valeur manquante. Les plages sont plausibles : Q10 de 0.42 à 6.04, MAT de −5.2 à 24.5 °C, MAP de 230 à 3386 mm/an, pH de 4.4 à 8.3, N515 de 837 à 9328 mg/kg (soit 0.8 à 9.3 g/kg d'azote).

## Étape 3 — Sauvegarde

**Décision :** on enregistre `data/dataset_v1.csv` (toutes les mesures, colonnes nettoyées).

In [6]:
# Sauvegarde de la version 1 du dataset
df_v1.to_csv("../data/dataset_v1.csv", index=False)

## Étape 4 — Version 2 : une ligne par site

**Objectif :** agréger les mesures faites au même point GPS (décision validée après `01_EDA.ipynb`, partie A).

**Raisonnement :** les variables explicatives sont identiques pour toutes les mesures d'un même point GPS, donc un modèle ne peut prédire que leur moyenne. La carte finale donne elle aussi une valeur par lieu. Enfin, agréger donne le même poids à chaque lieu ; sinon, un site à 9 parcelles pèse 9 fois plus qu'un site isolé.

**Décision :**
- une ligne par point GPS (`lat`, `lon`) ;
- `Q10` = moyenne arithmétique des Q10 du site (le choix le plus simple) ;
- `n_mesures` = nombre de mesures agrégées, pour garder cette information ;
- `site`, `source`, `study` = valeurs distinctes concaténées (ex. `paper_29 + paper_43`), pour garder la traçabilité ;
- les variables explicatives, identiques au sein d'un site (vérifié dans `01_EDA.ipynb`), sont gardées telles quelles.

In [7]:
# Fonction : concatène les valeurs distinctes d'une colonne texte (ex. "paper_29 + paper_43")
joindre = lambda s: " + ".join(s.unique())

In [8]:
# Agrégation : une ligne par point GPS (variables explicatives identiques), Q10 moyen et nombre de mesures
df_v2 = df_v1.groupby(["lat", "lon", "MAT_public", "MAP_public", "pH_public", "N515"], as_index=False).agg(
    site=("site", joindre), source=("source", joindre), study=("study", joindre), Q10=("Q10", "mean"), n_mesures=("Q10", "size"))

In [9]:
# Remise des colonnes dans l'ordre de la version 1 (n_mesures juste après Q10)
df_v2 = df_v2[["site", "source", "study", "Q10", "n_mesures", "lat", "lon", "MAT_public", "MAP_public", "pH_public", "N515"]]

In [10]:
# Vérification : nombre de sites et nombre total de mesures agrégées
print(len(df_v2), "sites |", df_v2.n_mesures.sum(), "mesures |", df_v2[["lat", "lon"]].duplicated().sum(), "point GPS en double")

131 sites | 173 mesures | 0 point GPS en double


**Constat :** 131 sites, soit un par point GPS, et 173 mesures au total : aucune mesure n'est perdue.

In [11]:
# Sauvegarde de la version 2 du dataset
df_v2.to_csv("../data/dataset_v2.csv", index=False)

## Résumé

- `data/dataset_v1.csv` : 173 lignes, 10 colonnes, aucune valeur manquante. Colonnes retirées : `Ts_max`, `Ts_mean`, `SWC_warm_rel`, `new`, `cluster` (origine ou sens inconnus).
- `data/dataset_v2.csv` : 131 lignes (une par site), avec `Q10` moyen et `n_mesures`. **C'est la version de référence pour la suite.**